In [ ]:
import pandas as pd
import glob
import os
from collections import Counter

DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"
files = glob.glob(DATA_ROOT + '/**/*.labeled', recursive=True)
print(f"Found {len(files)} files — processing in chunks to avoid RAM crash\n")

CHUNKSIZE = 10_000  # rows per chunk — safe for most laptops

# Accumulators — we collect STATISTICS not full data
total_rows       = 0
label_counter    = Counter()
detail_counter   = Counter()
scenario_counter = Counter()
col_names        = None
skipped          = []

for f in files:
    scenario = os.path.basename(os.path.dirname(os.path.dirname(f)))
    try:
        # Read header to get column names
        with open(f) as fh:
            for line in fh:
                if line.startswith('#fields'):
                    cols = [c.strip() for c in line.strip().split('\t')[1:]]
                    break

        if col_names is None:
            col_names = cols  # save once for reference

        # Read file in chunks — never loads full file into RAM
        for chunk in pd.read_csv(f, sep='\t', comment='#',
                                  low_memory=False, header=None,
                                  chunksize=CHUNKSIZE):
            chunk.columns = cols

            # Fix merged last column if present
            last_col = chunk.columns[-1]
            if '   ' in last_col:
                split_cols = chunk[last_col].str.strip().str.split(
                    r'\s{2,}', expand=True, n=2)
                split_cols.columns = ['tunnel_parents', 'label', 'detailed-label']
                chunk = chunk.drop(columns=[last_col])
                chunk = pd.concat([chunk, split_cols], axis=1)

            # Accumulate counts only — discard the chunk after
            total_rows += len(chunk)
            label_counter.update(chunk['label'].str.lower().str.strip().tolist())
            detail_counter.update(chunk['detailed-label'].tolist())
            scenario_counter[scenario] += len(chunk)

        print(f"  ✓ {scenario}")

    except Exception as e:
        skipped.append(scenario)
        print(f"  ✗ Skipped {scenario}: {e}")

# ── RESULTS ───────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"FULL DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total rows processed: {total_rows:,}")
print(f"Files skipped:        {len(skipped)}")

print(f"\n1. COLUMN NAMES ({len(col_names)} features):")
for i, c in enumerate(col_names):
    print(f"   [{i:02d}] {c}")

print(f"\n2. CLASS DISTRIBUTION (label — standardised to lowercase):")
for label, count in label_counter.most_common():
    pct = count / total_rows * 100
    print(f"   {label:<20} {count:>10,}  ({pct:.1f}%)")

benign_n  = label_counter.get('benign', 0)
attack_n  = label_counter.get('malicious', 0)
print(f"\n   Imbalance ratio — Benign : Malicious = 1 : {attack_n//benign_n if benign_n else 'N/A'}")

print(f"\n3. DETAILED ATTACK TYPES:")
for label, count in detail_counter.most_common():
    if label != '-':
        pct = count / total_rows * 100
        print(f"   {label:<35} {count:>8,}  ({pct:.1f}%)")

print(f"\n4. ROWS PER SCENARIO:")
for scenario, count in scenario_counter.most_common():
    print(f"   {scenario:<45} {count:>8,}")

if skipped:
    print(f"\n5. SKIPPED FILES ({len(skipped)}):")
    for s in skipped:
        print(f"   - {s}")